# 03 — Greedy Forward Selection  

**Purpose:**
1. Run the GFS algorithm to select the optimal ensemble team
2. Export `final_team_config.json` with selected models and WBF parameters

In [10]:
import os, json, itertools
import numpy as np

# --- CONFIGURATION ---
DATA_ROOT = r"C:\Users\dadab\Desktop\Clean Version\data"
OUTPUT_DIR = "outputs/"
GT_PATH = os.path.join(DATA_ROOT, "val/val_annotations.json")
BASELINE_NAME = "exp01_baseline_best.pth"

EVAL_IOU = 0.50
EVAL_CONF = 0.50

iou_candidates = [0.40, 0.45, 0.50, 0.55, 0.60, 0.65]
conf_candidates = [0.70, 0.75, 0.80, 0.85, 0.90]
param_grid = list(itertools.product(iou_candidates, conf_candidates))

## WBF and Evaluation Functions

In [11]:
def py_cpu_iou(boxA, boxB):
    xA, yA = max(boxA[0], boxB[0]), max(boxA[1], boxB[1])
    xB, yB = min(boxA[2], boxB[2]), min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0: return 0.0
    areaA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    areaB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (areaA + areaB - inter)

def wbf_strict(boxes, scores, labels, iou_thr, single_conf):
    """Strict Weighted Boxes Fusion: cluster by IoU, average by confidence."""
    if len(boxes) == 0: return [], [], []
    idx = np.argsort(scores)[::-1]
    boxes, scores, labels = boxes[idx], scores[idx], labels[idx]
    keep_b, keep_s, keep_l = [], [], []
    used = np.zeros(len(boxes), dtype=bool)

    for i in range(len(boxes)):
        if used[i]: continue
        cluster = [i]; used[i] = True
        for j in range(i+1, len(boxes)):
            if used[j] or labels[j] != labels[i]: continue
            if py_cpu_iou(boxes[i], boxes[j]) > iou_thr:
                cluster.append(j); used[j] = True

        max_s = np.max(scores[cluster])
        if len(cluster) >= 2 or (len(cluster) == 1 and max_s > single_conf):
            wb = np.average(boxes[cluster], axis=0, weights=scores[cluster])
            keep_b.append(wb); keep_s.append(max_s); keep_l.append(labels[i])

    return np.array(keep_b), np.array(keep_s), np.array(keep_l)

def evaluate_team(team, data_cache, gt_cache, img_ids, flat_gt, iou_thr, single_conf):
    """Evaluate a team using WBF fusion and return (found_set, fp_count, pred_count)."""
    gidx_map = {}; curr = 0
    for item in flat_gt:
        gidx_map.setdefault(item['img_id'], []).append(curr); curr += 1

    found, fp, n_preds = set(), 0, 0
    for img_id in img_ids:
        bl, sl, ll = [], [], []
        for m in team:
            if m in data_cache and img_id in data_cache[m]:
                d = data_cache[m][img_id]
                if len(d['b']) > 0:
                    bl.append(d['b']); sl.append(d['s']); ll.append(d['l'])
        if not bl: continue

        fb, fs, fl = wbf_strict(np.vstack(bl), np.concatenate(sl),
                                 np.concatenate(ll), iou_thr, single_conf)
        gts = gt_cache.get(img_id, [])
        matched = set()
        if len(fb) == 0: continue
        for k in range(len(fb)):
            if fs[k] < EVAL_CONF: continue
            n_preds += 1
            hit = False
            for j, gb in enumerate(gts):
                if j not in matched and py_cpu_iou(fb[k], gb) >= EVAL_IOU:
                    matched.add(j)
                    if img_id in gidx_map: found.add(gidx_map[img_id][j])
                    hit = True; break
            if not hit: fp += 1

    return found, fp, n_preds

## Load Data and Run GFS

In [12]:
# Load ground truth
with open(GT_PATH) as f: coco_gt = json.load(f)
img_ids = [i['id'] for i in coco_gt['images']]
gt_cache = {}; flat_gt = []
for ann in coco_gt['annotations']:
    iid = ann['image_id']
    box = [ann['bbox'][0], ann['bbox'][1],
           ann['bbox'][0]+ann['bbox'][2], ann['bbox'][1]+ann['bbox'][3]]
    gt_cache.setdefault(iid, []).append(box)
    flat_gt.append({'img_id': iid})

# Cache predictions
data_cache = {}; all_models = []
for f in os.listdir(OUTPUT_DIR):
    if f.startswith("val_results_") and f.endswith(".json"):
        mname = f.replace("val_results_", "").replace(".json", ".pth")
        all_models.append(mname)
        with open(os.path.join(OUTPUT_DIR, f)) as fh: raw = json.load(fh)
        mp = {}
        for r in raw:
            if r['score'] < 0.01: continue
            iid = r['image_id']
            mp.setdefault(iid, {'b':[],'s':[],'l':[]})
            x,y,w,h = r['bbox']
            mp[iid]['b'].append([x,y,x+w,y+h])
            mp[iid]['s'].append(r['score'])
            mp[iid]['l'].append(r['category_id'])
        data_cache[mname] = mp

print(f"Loaded {len(all_models)} models, {len(flat_gt)} GT annotations")

Loaded 20 models, 1830 GT annotations


In [4]:
# --- GFS ALGORITHM ---
team = [BASELINE_NAME]
remaining = [m for m in all_models if m != BASELINE_NAME]

# Baseline score
best_score, best_params = -9999, (0.5, 0.7)
for iou, conf in param_grid:
    found, fp, _ = evaluate_team(team, data_cache, gt_cache, img_ids, flat_gt, iou, conf)
    s = len(found) - fp * 0.1
    if s > best_score:
        best_score, best_params = s, (iou, conf)

current_score, current_params = best_score, best_params
print(f"Baseline: Score={current_score:.2f}, Params={current_params}")

step = 1
while remaining:
    print(f"\n--- STEP {step} (Team size: {len(team)}) ---")
    results = []
    for m in remaining:
        cand_team = team + [m]
        best_cs, best_cp = -9999, None
        for iou, conf in param_grid:
            f, fp, p = evaluate_team(cand_team, data_cache, gt_cache, img_ids, flat_gt, iou, conf)
            cs = len(f) - fp * 0.1
            if cs > best_cs:
                best_cs, best_cp = cs, (iou, conf)
                best_f = f
        results.append({'model': m, 'score': best_cs, 'gain': best_cs - current_score,
                        'params': best_cp, 'found': best_f})

    results.sort(key=lambda x: x['score'], reverse=True)
    for r in results[:5]:
        g = f"+{r['gain']:.2f}" if r['gain'] > 0 else f"{r['gain']:.2f}"
        print(f"  {r['model'][:35]:<35} | Gain: {g:<8} | Score: {r['score']:.2f}")

    w = results[0]
    if w['gain'] > 0:
        print(f"=> ACCEPTED: {w['model']} (IoU={w['params'][0]}, Conf={w['params'][1]})")
        team.append(w['model']); remaining.remove(w['model'])
        current_score, current_params = w['score'], w['params']
        step += 1
    else:
        print(f"=> STOPPING (best gain: {w['gain']:.2f})")
        break

# Export
config = {"optimal_iou": current_params[0], "optimal_conf": current_params[1],
          "final_score": current_score, "team": team}
with open(os.path.join(OUTPUT_DIR, "final_team_config.json"), 'w') as f:
    json.dump(config, f, indent=4)

print(f"\n=== FINAL TEAM ({len(team)} models) ===")
for i, m in enumerate(team): print(f"  {i+1}. {m}")
print(f"Score: {current_score:.2f} | IoU={current_params[0]}, Conf={current_params[1]}")

Baseline: Score=1433.30, Params=(0.5, 0.7)

--- STEP 1 (Team size: 1) ---
  exp02_hflip_best.pth                | Gain: +86.70   | Score: 1520.00
  exp03_vflip_best.pth                | Gain: +86.10   | Score: 1519.40
  exp12_griddistortion_best.pth       | Gain: +83.10   | Score: 1516.40
  exp07_randomresize_randomcrop_best. | Gain: +80.20   | Score: 1513.50
  exp04_rotate_best.pth               | Gain: +79.40   | Score: 1512.70
=> ACCEPTED: exp02_hflip_best.pth (IoU=0.6, Conf=0.75)

--- STEP 2 (Team size: 2) ---
  exp07_randomresize_randomcrop_best. | Gain: +18.50   | Score: 1538.50
  exp03_vflip_best.pth                | Gain: +18.00   | Score: 1538.00
  exp12_griddistortion_best.pth       | Gain: +15.50   | Score: 1535.50
  exp04_rotate_best.pth               | Gain: +14.90   | Score: 1534.90
  exp20_medianblur_best.pth           | Gain: +13.50   | Score: 1533.50
=> ACCEPTED: exp07_randomresize_randomcrop_best.pth (IoU=0.6, Conf=0.85)

--- STEP 3 (Team size: 3) ---
  exp18_motionbl